In [6]:
import requests
import pandas as pd

# --------------------------------------------------
# Locations
# --------------------------------------------------

locations = [
    {
        "name": "COAST centroid",
        "latitude": 29.55777238913,
        "longitude": -94.9841844555387
    },
    {
        "name": "EAST centroid",
        "latitude": 31.5748027652562,
        "longitude": -95.0097552766646
    },
    {
        "name": "FWEST centroid",
        "latitude": 31.41663562,
        "longitude": -103.0068717
    },
    {
        "name": "NCENT centroid",
        "latitude": 32.02335485,
        "longitude": -97.50734232
    },
    {
        "name": "NORTH centroid",
        "latitude": 33.66461115,
        "longitude": -98.05434614
    },
    {
            "name": "SCENT centroid",
            "latitude": 30.00621769,
            "longitude": -97.51005788
    },
    {
            "name": "SOUTH centroid",
            "latitude": 27.889562,
            "longitude":-97.98848239
    },
    {
            "name": "WEST centroid",
            "latitude": 31.3603825,
            "longitude": -100.3267206
    },
]

# --------------------------------------------------
# Open-Meteo API
# --------------------------------------------------

url = "https://archive-api.open-meteo.com/v1/archive"

all_data = []

for loc in locations:


    params = {
        "latitude": loc["latitude"],
        "longitude": loc["longitude"],

        # Add buffer around 2025 so conversion to Central Time
        # still gives the COMPLETE local calendar year
        "start_date": "2024-12-31",
        "end_date": "2026-01-01",

        "hourly": [
            "wind_speed_100m",
            "wind_direction_100m"
        ],
        "wind_speed_unit": "ms",
        # Keep API data in UTC
        "timezone": "UTC"
    }


    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()

    df = pd.DataFrame({
        "time_utc": data["hourly"]["time"],
        "wind_speed_100m_m_s": data["hourly"]["wind_speed_100m"],
        "wind_direction_100m_deg":
            data["hourly"]["wind_direction_100m"]
    })

    # Convert API timestamps to timezone-aware UTC
    df["time_utc"] = pd.to_datetime(
        df["time_utc"],
        utc=True
    )

    # Convert UTC -> Texas local time
    df["time_local"] = (
        df["time_utc"]
        .dt.tz_convert("America/Chicago")
    )

    # Keep only calendar year 2025 in TEXAS LOCAL TIME
    start_local = pd.Timestamp(
        "2025-01-01 00:00",
        tz="America/Chicago"
    )

    end_local = pd.Timestamp(
        "2026-01-01 00:00",
        tz="America/Chicago"
    )

    df = df[
        (df["time_local"] >= start_local) &
        (df["time_local"] < end_local)
    ].copy()

    df["location"] = loc["name"]
    df["latitude"] = loc["latitude"]
    df["longitude"] = loc["longitude"]

    all_data.append(df)

# Combine locations
wind_data = pd.concat(all_data, ignore_index=True)

wind_data = wind_data.sort_values(
    ["location", "time_utc"]
).reset_index(drop=True)

print(
    wind_data.groupby("location").size()
)

print(wind_data.head())
print(wind_data.groupby("location").size())

# Save
wind_data.to_csv("Wind_Data_2025.csv", index=False)

location
COAST centroid    8760
EAST centroid     8760
FWEST centroid    8760
NCENT centroid    8760
NORTH centroid    8760
SCENT centroid    8760
SOUTH centroid    8760
WEST centroid     8760
dtype: int64
                   time_utc  wind_speed_100m_m_s  wind_direction_100m_deg  \
0 2025-01-01 06:00:00+00:00                 8.19                       28   
1 2025-01-01 07:00:00+00:00                 8.14                       31   
2 2025-01-01 08:00:00+00:00                 8.02                       32   
3 2025-01-01 09:00:00+00:00                 8.00                       30   
4 2025-01-01 10:00:00+00:00                 8.39                       29   

                 time_local        location   latitude  longitude  
0 2025-01-01 00:00:00-06:00  COAST centroid  29.557772 -94.984184  
1 2025-01-01 01:00:00-06:00  COAST centroid  29.557772 -94.984184  
2 2025-01-01 02:00:00-06:00  COAST centroid  29.557772 -94.984184  
3 2025-01-01 03:00:00-06:00  COAST centroid  29.557772 -94.

In [5]:
import requests
import pandas as pd

locations = [
    {
        "name": "COAST centroid",
        "latitude": 29.55777238913,
        "longitude": -94.9841844555387
    },
    {
        "name": "EAST centroid",
        "latitude": 31.5748027652562,
        "longitude": -95.0097552766646
    },
    {
        "name": "FWEST centroid",
        "latitude": 31.41663562,
        "longitude": -103.0068717
    },
    {
        "name": "NCENT centroid",
        "latitude": 32.02335485,
        "longitude": -97.50734232
    },
    {
        "name": "NORTH centroid",
        "latitude": 33.66461115,
        "longitude": -98.05434614
    },
    {
        "name": "SCENT centroid",
        "latitude": 30.00621769,
        "longitude": -97.51005788
    },
    {
        "name": "SOUTH centroid",
        "latitude": 27.889562,
        "longitude": -97.98848239
    },
    {
        "name": "WEST centroid",
        "latitude": 31.3603825,
        "longitude": -100.3267206
    },
]

url = "https://archive-api.open-meteo.com/v1/archive"

all_data = []

for loc in locations:

    params = {
        "latitude": loc["latitude"],
        "longitude": loc["longitude"],

        # Add buffer around 2025 so conversion to Central Time
        # still gives the COMPLETE local calendar year
        "start_date": "2024-12-31",
        "end_date": "2026-01-01",

        "hourly": [
            "global_tilted_irradiance",
            "temperature_2m"
        ],

        # Fixed south-facing PV system
        "tilt": 30,
        "azimuth": 0,

        # Keep API data in UTC
        "timezone": "UTC"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()

    df = pd.DataFrame({
        "time_utc": data["hourly"]["time"],
        "GTI_W_m2": data["hourly"]["global_tilted_irradiance"],
        "temperature_C": data["hourly"]["temperature_2m"]
    })

    # Convert API timestamps to timezone-aware UTC
    df["time_utc"] = pd.to_datetime(
        df["time_utc"],
        utc=True
    )

    # Convert UTC -> Texas local time
    df["time_local"] = (
        df["time_utc"]
        .dt.tz_convert("America/Chicago")
    )

    # Keep only calendar year 2025 in TEXAS LOCAL TIME
    start_local = pd.Timestamp(
        "2025-01-01 00:00",
        tz="America/Chicago"
    )

    end_local = pd.Timestamp(
        "2026-01-01 00:00",
        tz="America/Chicago"
    )

    df = df[
        (df["time_local"] >= start_local) &
        (df["time_local"] < end_local)
    ].copy()

    df["location"] = loc["name"]
    df["latitude"] = loc["latitude"]
    df["longitude"] = loc["longitude"]

    all_data.append(df)


# --------------------------------------------------
# Combine all locations
# --------------------------------------------------

solar_data = pd.concat(
    all_data,
    ignore_index=True
)


# --------------------------------------------------
# Sort data
# --------------------------------------------------

solar_data = solar_data.sort_values(
    ["location", "time_utc"]
).reset_index(drop=True)


# --------------------------------------------------
# Check number of hours
# --------------------------------------------------

print(
    solar_data.groupby("location").size()
)


# --------------------------------------------------
# Save
# --------------------------------------------------

solar_data.to_csv(
    "Solar_Data_2025.csv",
    index=False
)

print(solar_data.head())

location
COAST centroid    8760
EAST centroid     8760
FWEST centroid    8760
NCENT centroid    8760
NORTH centroid    8760
SCENT centroid    8760
SOUTH centroid    8760
WEST centroid     8760
dtype: int64
                   time_utc  GTI_W_m2  temperature_C  \
0 2025-01-01 06:00:00+00:00       0.0           13.3   
1 2025-01-01 07:00:00+00:00       0.0           12.7   
2 2025-01-01 08:00:00+00:00       0.0           12.2   
3 2025-01-01 09:00:00+00:00       0.0           11.7   
4 2025-01-01 10:00:00+00:00       0.0           11.2   

                 time_local        location   latitude  longitude  
0 2025-01-01 00:00:00-06:00  COAST centroid  29.557772 -94.984184  
1 2025-01-01 01:00:00-06:00  COAST centroid  29.557772 -94.984184  
2 2025-01-01 02:00:00-06:00  COAST centroid  29.557772 -94.984184  
3 2025-01-01 03:00:00-06:00  COAST centroid  29.557772 -94.984184  
4 2025-01-01 04:00:00-06:00  COAST centroid  29.557772 -94.984184  


In [12]:
import time
import pandas as pd
import requests



# 1. Define your exact coordinates list
coordinates = [
    {"zone_id": 1, "latitude": 48.97844, "longitude": 11.86773},
    {"zone_id": 2, "latitude": 48.54391, "longitude": 8.505915},
    {"zone_id": 3, "latitude": 49.42351, "longitude": 6.748102},
    {"zone_id": 4, "latitude": 49.93534, "longitude": 7.077692},
    {"zone_id": 5, "latitude": 50.97741, "longitude": 9.238305},
    {"zone_id": 6, "latitude": 51.11904, "longitude": 10.80366},
    {"zone_id": 7, "latitude": 51.12265, "longitude": 13.45047},
    {"zone_id": 8, "latitude": 52.33513, "longitude": 13.38196},
    {"zone_id": 9, "latitude": 52.1026, "longitude": 11.48771},
    {"zone_id": 10, "latitude": 52.8243, "longitude": 9.585488},
    {"zone_id": 11, "latitude": 51.67001, "longitude": 6.572368},
    {"zone_id": 12, "latitude": 54.00346, "longitude": 9.696915},
    {"zone_id": 13, "latitude": 53.82148, "longitude": 13.11646},
]

# 2. Map API hourly variables to your attribute names
param_mapping = {
    "direct_normal_irradiance": "dni",
    "diffuse_radiation": "dhi",
    "shortwave_radiation": "ghi",
    "wind_speed_100m": "wind_speed",
    "temperature_2m": "air_temperature",
    "dew_point_2m": "dew_point",
    "relative_humidity_2m": "relative_humidity",
    "surface_pressure": "surface_pressure",
}

# 3. Open-Meteo Historical Archive endpoint
url = "https://archive-api.open-meteo.com/v1/archive"
all_dfs = []

print("Starting data download for 2024...")

for point in coordinates:
    print(
        f"Fetching Zone {point['zone_id']}: Lat {point['latitude']:.4f}, Lon {point['longitude']:.4f}"
    )

    params = {
        "latitude": point["latitude"],
        "longitude": point["longitude"],
        "start_date": "2024-01-01",
        "end_date": "2024-12-31",
        "hourly": list(param_mapping.keys()),
        "wind_speed_unit": "ms",  # m/s
        "timezone": "Europe/Berlin",  # Adjust timezone as needed
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    # Convert to DataFrame
    df_zone = pd.DataFrame(data["hourly"])
    df_zone["time"] = pd.to_datetime(df_zone["time"])

    # Rename variables to match your exact attribute names
    df_zone = df_zone.rename(columns=param_mapping)

    # Attach location identifiers
    df_zone["zone_id"] = point["zone_id"]
    df_zone["latitude"] = point["latitude"]
    df_zone["longitude"] = point["longitude"]

    all_dfs.append(df_zone)
    time.sleep(0.3)  # Brief pause between requests

# 4. Combine all locations into one master DataFrame
final_df = pd.concat(all_dfs, ignore_index=True)

# 5. Reorder columns
ordered_columns = [
    "zone_id",
    "latitude",
    "longitude",
    "time",
    "dni",
    "dhi",
    "ghi",
    "wind_speed",
    "air_temperature",
    "dew_point",
    "relative_humidity",
    "surface_pressure",
]
final_df = final_df[ordered_columns]

# Save to CSV
final_df.to_csv("Germany_zones_weather_2024.csv", index=False)

print("\nData extraction complete!")
print("DataFrame Shape:", final_df.shape)
print(final_df.head())

Starting data download for 2024...
Fetching Zone 1: Lat 48.9784, Lon 11.8677
Fetching Zone 2: Lat 48.5439, Lon 8.5059
Fetching Zone 3: Lat 49.4235, Lon 6.7481
Fetching Zone 4: Lat 49.9353, Lon 7.0777
Fetching Zone 5: Lat 50.9774, Lon 9.2383
Fetching Zone 6: Lat 51.1190, Lon 10.8037
Fetching Zone 7: Lat 51.1227, Lon 13.4505
Fetching Zone 8: Lat 52.3351, Lon 13.3820
Fetching Zone 9: Lat 52.1026, Lon 11.4877
Fetching Zone 10: Lat 52.8243, Lon 9.5855
Fetching Zone 11: Lat 51.6700, Lon 6.5724
Fetching Zone 12: Lat 54.0035, Lon 9.6969
Fetching Zone 13: Lat 53.8215, Lon 13.1165

Data extraction complete!
DataFrame Shape: (114192, 12)
   zone_id  latitude  longitude                time  dni  dhi  ghi  \
0        1  48.97844   11.86773 2024-01-01 00:00:00  0.0  0.0  0.0   
1        1  48.97844   11.86773 2024-01-01 01:00:00  0.0  0.0  0.0   
2        1  48.97844   11.86773 2024-01-01 02:00:00  0.0  0.0  0.0   
3        1  48.97844   11.86773 2024-01-01 03:00:00  0.0  0.0  0.0   
4        1  48.

In [10]:
import sys
sys.path.append('../src')

In [13]:
from energiapy.conversion.photovoltaic import solar_power_output
from energiapy.conversion.windmill import wind_power_output
from energiapy.utils.math_utils import normalize
import numpy as np
import pandas as pd

# 1. Load the weather dataset
weather_df = pd.read_csv("Germany_zones_weather_2024.csv")
weather_df["time"] = pd.to_datetime(weather_df["time"])

solar_dict = {}
wind_dict = {}
timestamps = None

# 2. Iterate through each zone (1 to 13)
zones = sorted(weather_df["zone_id"].unique())

for zone in zones:
    print(f"Processing Zone {zone}...")

    # Filter for the current zone and set datetime index
    zone_data = (
        weather_df[weather_df["zone_id"] == zone]
        .copy()
        .sort_values("time")
        .set_index("time")
    )

    if timestamps is None:
        timestamps = zone_data.index

    coord = (zone_data["latitude"].iloc[0], zone_data["longitude"].iloc[0])

    # 3. Solar Power & Normalization (flatten to 1D)
    solar = solar_power_output(
        data=zone_data,
        coord=coord,
        sam="cecmod",
        module_params="Canadian_Solar_Inc__CS5P_220M",
        inverter="cecinverter",
        inverter_params="ABB__MICRO_0_25_I_OUTD_US_208__208V_",
        temperature_params="open_rack_glass_glass",
        aoi_model="no_loss",
        ac_model="sandia",
        spectral_model="no_loss",
    )
    solar_norm = normalize(solar)
    # Convert/flatten to 1D array
    solar_dict[f"Zone_{zone}"] = np.array(solar_norm).ravel()

    # 4. Wind Power & Normalization (flatten to 1D)
    wind = wind_power_output(
        data=zone_data,
        roughness_length=0.1,
        turbine_type="V100/1800",
        hub_height=92,
        wind_speed_model="logarithmic",
        density_model="ideal_gas",
        temperature_model="linear_gradient",
        power_output_model="power_coefficient_curve",
        density_correction=True,
        obstacle_height=0,
        observation_height=10,
    )
    wind_norm = normalize(wind)
    # Convert/flatten to 1D array
    wind_dict[f"Zone_{zone}"] = np.array(wind_norm).ravel()

# ---------------------------------------------------------
# 5. Build and Save the Two 11-Column CSVs
# ---------------------------------------------------------
# File 1: Solar Capacity Factor CSV
df_solar_final = pd.DataFrame(solar_dict, index=timestamps)
df_solar_final.index.name = "time"
df_solar_final.to_csv("Germany_solar_capacity_factor_2024.csv")

# File 2: Wind Capacity Factor CSV
df_wind_final = pd.DataFrame(wind_dict, index=timestamps)
df_wind_final.index.name = "time"
df_wind_final.to_csv("Germany_wind_capacity_factor_2024.csv")

print("\nDone! Preview:")
print("Solar Head:\n", df_solar_final.head())
print("\nWind Head:\n", df_wind_final.head())

Processing Zone 1...
Processing Zone 2...
Processing Zone 3...
Processing Zone 4...
Processing Zone 5...
Processing Zone 6...
Processing Zone 7...
Processing Zone 8...
Processing Zone 9...
Processing Zone 10...
Processing Zone 11...
Processing Zone 12...
Processing Zone 13...

Done! Preview:
Solar Head:
                      Zone_1  Zone_2  Zone_3  Zone_4  Zone_5  Zone_6  Zone_7  \
time                                                                          
2024-01-01 00:00:00     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
2024-01-01 01:00:00     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
2024-01-01 02:00:00     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
2024-01-01 03:00:00     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
2024-01-01 04:00:00     0.0     0.0     0.0     0.0     0.0     0.0     0.0   

                     Zone_8  Zone_9  Zone_10  Zone_11  Zone_12  Zone_13  
time                                                              

## UK Branch

In [14]:
import time
import pandas as pd
import requests



# 1. Define your exact coordinates list
coordinates = [
    {"zone_id": 1, "latitude": 54.94098, "longitude": -1.65748},
    {"zone_id": 2, "latitude": 48.54391, "longitude": -2.67833},
    {"zone_id": 3, "latitude": 49.42351, "longitude": -1.18605},
    {"zone_id": 4, "latitude": 49.93534, "longitude": -0.80288},
    {"zone_id": 5, "latitude": 50.97741, "longitude": -2.1468},
    {"zone_id": 6, "latitude": 51.11904, "longitude": 0.61525},
    {"zone_id": 7, "latitude": 51.12265, "longitude": -0.14967},
    {"zone_id": 8, "latitude": 52.33513, "longitude": -0.47567},
    {"zone_id": 9, "latitude": 52.1026, "longitude": -3.16825},
    {"zone_id": 10, "latitude": 52.8243, "longitude": -3.71947},
    {"zone_id": 11, "latitude": 51.67001, "longitude": -3.80442},
]

# 2. Map API hourly variables to your attribute names
param_mapping = {
    "direct_normal_irradiance": "dni",
    "diffuse_radiation": "dhi",
    "shortwave_radiation": "ghi",
    "wind_speed_100m": "wind_speed",
    "temperature_2m": "air_temperature",
    "dew_point_2m": "dew_point",
    "relative_humidity_2m": "relative_humidity",
    "surface_pressure": "surface_pressure",
}

# 3. Open-Meteo Historical Archive endpoint
url = "https://archive-api.open-meteo.com/v1/archive"
all_dfs = []

print("Starting data download for 2024...")

for point in coordinates:
    print(
        f"Fetching Zone {point['zone_id']}: Lat {point['latitude']:.4f}, Lon {point['longitude']:.4f}"
    )

    params = {
        "latitude": point["latitude"],
        "longitude": point["longitude"],
        "start_date": "2024-01-01",
        "end_date": "2024-12-31",
        "hourly": list(param_mapping.keys()),
        "wind_speed_unit": "ms",  # m/s
        "timezone": "Europe/London",  # Adjust timezone as needed
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    # Convert to DataFrame
    df_zone = pd.DataFrame(data["hourly"])
    df_zone["time"] = pd.to_datetime(df_zone["time"])

    # Rename variables to match your exact attribute names
    df_zone = df_zone.rename(columns=param_mapping)

    # Attach location identifiers
    df_zone["zone_id"] = point["zone_id"]
    df_zone["latitude"] = point["latitude"]
    df_zone["longitude"] = point["longitude"]

    all_dfs.append(df_zone)
    time.sleep(0.3)  # Brief pause between requests

# 4. Combine all locations into one master DataFrame
final_df = pd.concat(all_dfs, ignore_index=True)

# 5. Reorder columns
ordered_columns = [
    "zone_id",
    "latitude",
    "longitude",
    "time",
    "dni",
    "dhi",
    "ghi",
    "wind_speed",
    "air_temperature",
    "dew_point",
    "relative_humidity",
    "surface_pressure",
]
final_df = final_df[ordered_columns]

# Save to CSV
final_df.to_csv("UK_zones_weather_2024.csv", index=False)

print("\nData extraction complete!")
print("DataFrame Shape:", final_df.shape)
print(final_df.head())

Starting data download for 2024...
Fetching Zone 1: Lat 54.9410, Lon -1.6575
Fetching Zone 2: Lat 48.5439, Lon -2.6783
Fetching Zone 3: Lat 49.4235, Lon -1.1861
Fetching Zone 4: Lat 49.9353, Lon -0.8029
Fetching Zone 5: Lat 50.9774, Lon -2.1468
Fetching Zone 6: Lat 51.1190, Lon 0.6152
Fetching Zone 7: Lat 51.1227, Lon -0.1497
Fetching Zone 8: Lat 52.3351, Lon -0.4757
Fetching Zone 9: Lat 52.1026, Lon -3.1683
Fetching Zone 10: Lat 52.8243, Lon -3.7195
Fetching Zone 11: Lat 51.6700, Lon -3.8044

Data extraction complete!
DataFrame Shape: (96624, 12)
   zone_id  latitude  longitude                time  dni  dhi  ghi  \
0        1  54.94098   -1.65748 2024-01-01 00:00:00  0.0  0.0  0.0   
1        1  54.94098   -1.65748 2024-01-01 01:00:00  0.0  0.0  0.0   
2        1  54.94098   -1.65748 2024-01-01 02:00:00  0.0  0.0  0.0   
3        1  54.94098   -1.65748 2024-01-01 03:00:00  0.0  0.0  0.0   
4        1  54.94098   -1.65748 2024-01-01 04:00:00  0.0  0.0  0.0   

   wind_speed  air_temper

In [15]:
import sys
sys.path.append('../src')

In [16]:
from energiapy.conversion.photovoltaic import solar_power_output
from energiapy.conversion.windmill import wind_power_output
from energiapy.utils.math_utils import normalize
import numpy as np
import pandas as pd

# 1. Load the weather dataset
weather_df = pd.read_csv("UK_zones_weather_2024.csv")
weather_df["time"] = pd.to_datetime(weather_df["time"])

solar_dict = {}
wind_dict = {}
timestamps = None

# 2. Iterate through each zone (1 to 13)
zones = sorted(weather_df["zone_id"].unique())

for zone in zones:
    print(f"Processing Zone {zone}...")

    # Filter for the current zone and set datetime index
    zone_data = (
        weather_df[weather_df["zone_id"] == zone]
        .copy()
        .sort_values("time")
        .set_index("time")
    )

    if timestamps is None:
        timestamps = zone_data.index

    coord = (zone_data["latitude"].iloc[0], zone_data["longitude"].iloc[0])

    # 3. Solar Power & Normalization (flatten to 1D)
    solar = solar_power_output(
        data=zone_data,
        coord=coord,
        sam="cecmod",
        module_params="Canadian_Solar_Inc__CS5P_220M",
        inverter="cecinverter",
        inverter_params="ABB__MICRO_0_25_I_OUTD_US_208__208V_",
        temperature_params="open_rack_glass_glass",
        aoi_model="no_loss",
        ac_model="sandia",
        spectral_model="no_loss",
    )
    solar_norm = normalize(solar)
    # Convert/flatten to 1D array
    solar_dict[f"Zone_{zone}"] = np.array(solar_norm).ravel()

    # 4. Wind Power & Normalization (flatten to 1D)
    wind = wind_power_output(
        data=zone_data,
        roughness_length=0.1,
        turbine_type="V100/1800",
        hub_height=92,
        wind_speed_model="logarithmic",
        density_model="ideal_gas",
        temperature_model="linear_gradient",
        power_output_model="power_coefficient_curve",
        density_correction=True,
        obstacle_height=0,
        observation_height=10,
    )
    wind_norm = normalize(wind)
    # Convert/flatten to 1D array
    wind_dict[f"Zone_{zone}"] = np.array(wind_norm).ravel()

# ---------------------------------------------------------
# 5. Build and Save the Two 11-Column CSVs
# ---------------------------------------------------------
# File 1: Solar Capacity Factor CSV
df_solar_final = pd.DataFrame(solar_dict, index=timestamps)
df_solar_final.index.name = "time"
df_solar_final.to_csv("UK_solar_capacity_factor_2024.csv")

# File 2: Wind Capacity Factor CSV
df_wind_final = pd.DataFrame(wind_dict, index=timestamps)
df_wind_final.index.name = "time"
df_wind_final.to_csv("UK_wind_capacity_factor_2024.csv")

print("\nDone! Preview:")
print("Solar Head:\n", df_solar_final.head())
print("\nWind Head:\n", df_wind_final.head())

Processing Zone 1...
Processing Zone 2...
Processing Zone 3...
Processing Zone 4...
Processing Zone 5...
Processing Zone 6...
Processing Zone 7...
Processing Zone 8...
Processing Zone 9...
Processing Zone 10...
Processing Zone 11...

Done! Preview:
Solar Head:
                      Zone_1  Zone_2  Zone_3  Zone_4  Zone_5  Zone_6  Zone_7  \
time                                                                          
2024-01-01 00:00:00     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
2024-01-01 01:00:00     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
2024-01-01 02:00:00     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
2024-01-01 03:00:00     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
2024-01-01 04:00:00     0.0     0.0     0.0     0.0     0.0     0.0     0.0   

                     Zone_8  Zone_9  Zone_10  Zone_11  
time                                                   
2024-01-01 00:00:00     0.0     0.0      0.0      0.0  
2024-01-01 01:00